In [21]:
import pandas as pd
import matplotlib.pyplot as plt

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
# Load the datasest
file_path = '/content/drive/MyDrive/COE 547 - Final Project/ECG_10s_2500.csv'
df = pd.read_csv(file_path)

In [24]:
# Number of patients
patients_number = df['patient_id'].nunique()
print(f'The total number of patients is: {patients_number}')

The total number of patients is: 9


In [25]:
# Group by patient_id and label to count occurrences
patient_label_counts = df.groupby(['patient_id', 'label']).size().unstack(fill_value=0)

# Rename columns for clarity
patient_label_counts = patient_label_counts.rename(columns={0: 'non_hypo', 1: 'hypo'})

print("Number of hypo and non-hypo cases per patient:")
print(patient_label_counts)

Number of hypo and non-hypo cases per patient:
label       non_hypo  hypo
patient_id                
1                515    54
2               4026   372
3               1723     0
4               1968   358
5                 44     0
6               5621     0
7              14360     3
8              14886  1674
9                 26     0


In [26]:
# Dataset balance
df["label"].value_counts()

,count
label,
0,43169
1,2461


In [27]:
cols_to_drop = ["day", "count", "glucose_values", "patient_id"]
existing = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=existing)

In [28]:
import numpy as np

ecg_cols = [c for c in df.columns if c != "label"]
X = df[ecg_cols].to_numpy(dtype=np.float32)
y = df["label"].to_numpy(dtype=np.int32)

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [30]:
input_length = X_train.shape[1]   # automatically 2500 or 500

X_train = X_train.reshape(-1, input_length, 1)
X_val   = X_val.reshape(-1, input_length, 1)
X_test  = X_test.reshape(-1, input_length, 1)

In [31]:
X_pos = X_train[y_train == 1]
X_neg = X_train[y_train == 0]

print("Before balancing:", len(X_pos), "hypo |", len(X_neg), "normal")

Before balancing: 1723 hypo | 30218 normal


In [32]:
import numpy as np

def slice_with_overlap(X, y, window_size=500, overlap=0.5):
    """
    X: (N, T, 1) array, T = original length (e.g. 2500)
    y: (N,) array of labels
    window_size: length of each sliced window (e.g. 500 for 2s)
    overlap: fraction, 0.5 => 50% overlap, 0.0 => no overlap

    Returns:
      X_new: (M, window_size, 1)
      y_new: (M,)
    """
    step = int(window_size * (1 - overlap))
    if step <= 0:
        raise ValueError("overlap too large: step <= 0")

    X_new = []
    y_new = []

    for i in range(len(X)):
        signal = X[i].squeeze()          # (T,)
        T = len(signal)

        # slide window
        for start in range(0, T - window_size + 1, step):
            end = start + window_size
            seg = signal[start:end]
            X_new.append(seg.reshape(window_size, 1))
            y_new.append(y[i])

    return np.array(X_new), np.array(y_new)

In [33]:
window_size = 500   # 2-second windows

X_train_win, y_train_win = slice_with_overlap(
    X_train, y_train,
    window_size=window_size,
    overlap=0.75        # 50% overlap on train
)

X_val_win, y_val_win = slice_with_overlap(
    X_val, y_val,
    window_size=window_size,
    overlap=0.0        # NO overlap on val
)

X_test_win, y_test_win = slice_with_overlap(
    X_test, y_test,
    window_size=window_size,
    overlap=0.0        # NO overlap on test
)

In [34]:
import numpy as np

def show_balance(name, y):
    total = len(y)
    pos = np.sum(y == 1)
    neg = np.sum(y == 0)
    print(f"===== {name} =====")
    print(f"Total samples: {total}")
    print(f"Non-Hypo (0): {neg}")
    print(f"Hypo     (1): {pos}")
    print(f"Ratio (hypo %): {pos/total:.4f}")
    print()

show_balance("TRAIN (after slicing)", y_train_win)
show_balance("VAL   (after slicing)", y_val_win)
show_balance("TEST  (after slicing)", y_test_win)

===== TRAIN (after slicing) =====
Total samples: 542997
Non-Hypo (0): 513706
Hypo     (1): 29291
Ratio (hypo %): 0.0539

===== VAL   (after slicing) =====
Total samples: 34220
Non-Hypo (0): 32375
Hypo     (1): 1845
Ratio (hypo %): 0.0539

===== TEST  (after slicing) =====
Total samples: 34225
Non-Hypo (0): 32380
Hypo     (1): 1845
Ratio (hypo %): 0.0539



In [35]:
X_flat = X_train_win.reshape(len(X_train_win), -1)
y_flat = y_train_win

minority = np.sum(y_flat == 1)
target_majority = minority * 4          # make hypo = 20%
print("Hypo:", minority)
print("Target majority:", target_majority)

from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(
    sampling_strategy={0: target_majority},   # keep class 0 at desired size
    random_state=42
)

X_bal_flat, y_bal = rus.fit_resample(X_flat, y_flat)

X_train_bal = X_bal_flat.reshape(-1, 500, 1)
y_train_bal = y_bal

show_balance("TRAIN AFTER BALANCING", y_train_bal)

Hypo: 29291
Target majority: 117164
===== TRAIN AFTER BALANCING =====
Total samples: 146455
Non-Hypo (0): 117164
Hypo     (1): 29291
Ratio (hypo %): 0.2000



In [36]:
from keras import layers, models

def build_model(input_length):
    inp = layers.Input(shape=(input_length,1))

    # Block 1
    x = layers.Conv1D(32, 15, padding='same', activation='relu')(inp)
    x = layers.MaxPooling1D(2)(x)
    x = layers.BatchNormalization()(x)

    # Block 2
    x = layers.Conv1D(64, 11, padding='same', activation='relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.BatchNormalization()(x)

    # Block 3
    x = layers.Conv1D(128, 7, padding='same', activation='relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.BatchNormalization()(x)

    # LSTM
    # Added return_sequences=True to the first Bidirectional LSTM layer
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(32))(x)
    x = layers.Dropout(0.3)(x)


    # Dense head
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=inp, outputs=out)

model = build_model(window_size)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 500, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 500, 32)        │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 250, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 250, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 250, 64)        │        22,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 125, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 125, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 125, 128)       │        57,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 62, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 62, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 62, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 223,617 (873.50 KB)

 Trainable params: 223,169 (871.75 KB)

 Non-trainable params: 448 (1.75 KB)

In [37]:
pos = y_train_bal.sum()
neg = len(y_train_bal) - pos
class_weights = {0: 1.0, 1: neg / pos}
print(class_weights)

{0: 1.0, 1: np.float64(4.0)}


In [38]:
import tensorflow as tf

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.AUC(name='roc_auc'),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

In [ ]:
history = model.fit(
    X_train_win, y_train_win,
    validation_data=(X_val_win, y_val_win),
    epochs=40,
    batch_size=128,
    class_weight=class_weights
)

Epoch 1/40
4242/4243 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.3929 - pr_auc: 0.2780 - precision: 0.2911 - recall: 0.3459 - roc_auc: 0.8532

In [ ]:
import numpy as np
from sklearn.metrics import (
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)

# ---- 1) Get model probabilities on test ----
y_test_pred_prob = model.predict(X_test_win).ravel()

# ---- 2) Find best threshold (F1-optimal) ----
precision_vals, recall_vals, thresholds_pr = precision_recall_curve(y_test_win, y_test_pred_prob)
f1_vals = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-7)

best_idx = np.argmax(f1_vals)
best_threshold = thresholds_pr[best_idx]
print("Best threshold (F1):", best_threshold)

# ---- 3) Convert probabilities → predicted labels ----
y_test_pred = (y_test_pred_prob >= best_threshold).astype(int)

# ---- 4) Confusion Matrix ----
cm = confusion_matrix(y_test_win, y_test_pred)
print("Confusion Matrix:\n", cm)

# ---- 5) Classification Report ----
print("\nClassification Report (test):")
print(classification_report(y_test_win, y_test_pred, digits=4))

# ---- 6) Sensitivity, Specificity, F1 ----
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn + 1e-7)   # recall for hypo
specificity = tn / (tn + fp + 1e-7)

print("Sensitivity (Hypo Recall):", sensitivity)
print("Specificity:", specificity)

precision = tp / (tp + fp + 1e-7)
recall    = sensitivity
f1        = 2 * precision * recall / (precision + recall + 1e-7)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
